In [1]:
# !pip install ucimlrepo

In [2]:
from ucimlrepo import fetch_ucirepoimport pandas as pdimport matplotlib.pyplot as pltimport numpy as npimport seaborn as snsimport warningsfrom sklearn.preprocessing import MinMaxScalerfrom sklearn.decomposition import PCA # to visualize high dimensional datafrom sklearn.manifold import TSNEfrom sklearn.metrics import adjusted_rand_score, normalized_mutual_info_scorefrom sklearn.ensemble import RandomForestClassifierfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import accuracy_score, classification_reportwarnings.filterwarnings('ignore')

## The DATA

In [3]:
# fetch datasetbank_marketing = fetch_ucirepo(id=222)# data (as pandas dataframes)X = bank_marketing.data.featuresy = bank_marketing.data.targets# metadataprint(bank_marketing.metadata)# variable informationprint(bank_marketing.variables)

{'uci_id': 222, 'name': 'Bank Marketing', 'repository_url': 'https://archive.ics.uci.edu/dataset/222/bank+marketing', 'data_url': 'https://archive.ics.uci.edu/static/public/222/data.csv', 'abstract': 'The data is related with direct marketing campaigns (phone calls) of a Portuguese banking institution. The classification goal is to predict if the client will subscribe a term deposit (variable y).', 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 45211, 'num_features': 16, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Occupation', 'Marital Status', 'Education Level'], 'target_col': ['y'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2014, 'last_updated': 'Fri Aug 18 2023', 'dataset_doi': '10.24432/C5K306', 'creators': ['S. Moro', 'P. Rita', 'P. Cortez'], 'intro_paper': {'ID': 277, 'type': 'NATIVE', 'title': 'A data-driven approach to predict the s

## Preprocessing

In [4]:
X.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN


In [5]:
y.dtypes

y    object
dtype: object

we have to convert our label into numerical label (right now its categorical as can be seen above)

In [6]:
X.dtypes

age             int64
job            object
marital        object
education      object
default        object
balance         int64
housing        object
loan           object
contact        object
day_of_week     int64
month          object
duration        int64
campaign        int64
pdays           int64
previous        int64
poutcome       object
dtype: object

so in addition to the labels, we have a lot of features we have to convert from categorical to numerical

## missing values

In [7]:
## MISSING DATAtotal = X.isnull().sum().sort_values(ascending=False)percent = (X.isnull().sum() / X.isnull().count()).sort_values(ascending=False).round(4)missing_data = pd.concat([total, percent], axis=1, keys=["Total", 'Percent'])missing_data.head(20)

,Total,Percent
poutcome,36959,0.8175
contact,13020,0.2880
education,1857,0.0411
job,288,0.0064
age,0,0.0000
marital,0,0.0000
default,0,0.0000
balance,0,0.0000
housing,0,0.0000
loan,0,0.0000


before decoding the categorical features, lets first handle the missing values

as can be seen from the table above, we have missing values in 4 features:

1. poutcome: missing so many values, so the best option is to just treat the missing as a category (we just fill them with 'unknown')

2. contact: for this one we only have 2 unique values ('cellular' and 'telephone'), we can't really replace the missing with any of them sense theyre missing close to a third of the values, so we just fill it with 'missing'

3. education: we're goning to fill the data with the most frequent value, becuase we're only missing about 5% of them, and also even in the real world, for the education feature of actual people this would probably work the best out of all the methods we can impelement.

4. job: the missing percentage is too small to have an impact on the performnace of our model, so we're going to do the same thing as the 'education'

In [8]:
X['poutcome'] = X['poutcome'].fillna('unknown')X['contact'] = X['contact'].fillna('unknown')

In [9]:
X['education'] = X['education'].fillna(X['education'].mode().item())X['job'] = X['job'].fillna(X['job'].mode().item())

lets check if it worked

In [10]:
## MISSING DATAtotal = X.isnull().sum().sort_values(ascending=False)percent = (X.isnull().sum() / X.isnull().count()).sort_values(ascending=False).round(4)missing_data = pd.concat([total, percent], axis=1, keys=["Total", 'Percent'])missing_data.head(20)

,Total,Percent
age,0,0.0
job,0,0.0
marital,0,0.0
education,0,0.0
default,0,0.0
balance,0,0.0
housing,0,0.0
loan,0,0.0
contact,0,0.0
day_of_week,0,0.0


it did, so now we can go to the 'encoding' step, which means we want to convert the categorical values to numerical

## Categorical to numerical

the binary values can just get replaced with 1 and 0
the binary values as mentioned in the website provided in the assignment instructions are:

1.loan

2.housing

3.default

In [11]:
X['loan'] = X['loan'].map({'yes': 1, 'no': 0})X['housing'] = X['housing'].map({'yes': 1, 'no': 0})X['default'] = X['default'].map({'yes': 1, 'no': 0})